# 12. LangGraph for Stateful Data Science AI Applications

**Difficulty:** Expert | **Time:** 3-4 hours | **Prerequisites:** Notebooks 01-11

By the end of this notebook you will be able to:

- Build a stateful graph-based workflow with LangGraph
- Use conditional routing to direct questions to the right handler
- Add validation loops and human-in-the-loop steps
- Combine RAG, tools, and LLM reasoning in a single graph
- Compare LangChain chains with LangGraph workflows

---

## 1. Why LangGraph?

Simple LangChain chains follow a **fixed pipeline**:

```
Input  -->  Prompt  -->  Model  -->  Parser  -->  Output
```

This works well when the workflow is linear. But real Data Science assistants need to:

- **Classify** the question first (conceptual? numerical? database?)
- **Route** to different handlers (RAG, tool, direct LLM)
- **Loop** if the answer needs refinement
- **Remember** context across steps

LangGraph lets you model this as a **graph** of nodes and edges.

### LangGraph Architecture

```mermaid
graph TD
    START([START]) --> Classify[Classify Question]
    Classify -->|conceptual| RAG[RAG Retriever]
    Classify -->|numerical| Tool[Calculation Tool]
    Classify -->|general| LLM[Direct LLM]
    RAG --> Validate[Validate Answer]
    Tool --> Validate
    LLM --> Validate
    Validate -->|pass| END([END])
    Validate -->|needs work| Classify
```

---

## 2. Setup


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

# Check API key
if os.getenv('OPENAI_API_KEY'):
    print('OpenAI API key found')
else:
    print('Warning: No OPENAI_API_KEY. API examples will not work.')
    print('Set it in .env or as an environment variable.')

In [ ]:
# Install: pip install langgraph langchain-core langchain-openai langchain-ollama langchain-chroma
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_openai import ChatOpenAI
print('Core imports successful')

In [ ]:
# LangGraph imports
from langgraph.graph import StateGraph, START, END
print('LangGraph imported successfully')

In [ ]:
# Optional Ollama support
ollama_available = False
try:
    from langchain_ollama import ChatOllama
    import socket
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    result = s.connect_ex(('127.0.0.1', 11434))
    s.close()
    if result == 0:
        ollama_available = True
        print('Ollama detected!')
    else:
        print('Ollama not running. Local examples will be skipped.')
        print('Start with: ollama serve')
except Exception:
    print('Ollama not available.')

In [ ]:
# Create LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print('LLM ready')

---

## 3. Graph Concepts

### The simplest possible graph

A LangGraph workflow has three core concepts:

| Concept | Description |
|---------|-------------|
| **State** | A dictionary that flows through the graph |
| **Node** | A function that reads state, does work, returns updates |
| **Edge** | A connection between nodes (can be conditional) |

Let us build the simplest graph first:

In [ ]:
# Define a simple state type (just a dict for now)

# Node: takes state, returns partial update
def greeting_node(state):
    name = state.get('name', 'Student')
    return {'message': f'Hello, {name}! Welcome to Data Science.'}

# Build the graph
builder = StateGraph(dict)
builder.add_node('greet', greeting_node)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)

# Compile and run
graph = builder.compile()
result = graph.invoke({'name': 'Alice'})
print(result['message'])

### What happened?

1. We defined a **state** (`{'name': 'Alice'}`)
2. We created a **node** (`greeting_node`) that reads the state and returns an update
3. We built a **graph** with one node, connected from START to END
4. We **invoked** the graph with initial state
5. The graph returned the updated state with the `message` field

---

## 4. Data Science Research Assistant Graph

Now let us build something useful: a graph that classifies Data Science questions
and routes them to the appropriate handler.

```mermaid
graph TD
    START([START]) --> Classify[Classify Question]
    Classify -->|conceptual| RAG[RAG: Search Knowledge Base]
    Classify -->|numerical| Tool[Tool: Calculate Stats]
    Classify -->|general| LLM[Direct LLM Answer]
    RAG --> Combine[Combine Results]
    Tool --> Combine
    LLM --> Combine
    Combine --> Validate[Validate Answer]
    Validate -->|pass| END([END])
    Validate -->|improve| LLM
```

---

## 5. State Design

The state carries information through the graph. Each node reads from and writes to this shared state.

```python
# Our state fields:
state = {
    'question': '',       # Original question
    'route': '',          # Classification: 'conceptual' | 'numerical' | 'general'
    'context': '',        # Retrieved documents (for RAG)
    'tool_result': '',    # Tool output (for calculations)
    'answer': '',         # Generated answer
    'validated': False,   # Has the answer been validated?
    'attempts': 0,        # Number of refinement attempts
}
```

---

## 6. Defining Nodes


In [ ]:
from langchain_core.messages import HumanMessage

# ---- Knowledge base for RAG ----
ds_knowledge = {
    'regression': 'Regression predicts continuous values. Linear regression fits a straight line. Logistic regression is for classification using a sigmoid function.',
    'classification': 'Classification assigns categories. Common algorithms: Decision Trees, Random Forest, SVM, KNN. Use accuracy, precision, recall, F1.',
    'clustering': 'Clustering groups similar data points. K-Means assigns points to nearest centroid. DBSCAN finds dense regions.',
    'evaluation': 'Model evaluation uses train/test splits. Cross-validation provides robust estimates. Metrics: accuracy, precision, recall, F1, AUC-ROC.',
}

def classify_node(state):
    """Classify the question into a category."""
    question = state['question'].lower()

    # Simple keyword-based classification (in production, use the LLM)
    conceptual_keywords = ['what is', 'explain', 'describe', 'how does', 'tell me about']
    numerical_keywords = ['calculate', 'compute', 'mean', 'sum', 'average', 'statistics']

    if any(kw in question for kw in numerical_keywords):
        route = 'numerical'
    elif any(kw in question for kw in conceptual_keywords):
        route = 'conceptual'
    else:
        route = 'general'

    print(f'  [Classify] Route: {route}')
    return {'route': route, 'attempts': 0}

# Test
result = classify_node({'question': 'Explain Random Forest', 'route': '', 'context': '', 'tool_result': '', 'answer': ''})
print(f'Route: {result}')

In [ ]:
def rag_node(state):
    """Search knowledge base for relevant information."""
    question = state['question'].lower()
    context_parts = []

    # Simple keyword search (in production, use vector store + retriever)
    for topic, info in ds_knowledge.items():
        if topic in question or any(word in question for word in topic.split()):
            context_parts.append(info)

    # Also search for related terms
    if not context_parts:
        for topic, info in ds_knowledge.items():
            question_words = set(question.split())
            topic_words = set(topic.split())
            if question_words & topic_words:
                context_parts.append(info)

    if not context_parts:
        context_parts = list(ds_knowledge.values())[:2]  # fallback: take first two

    context = ' '.join(context_parts)
    print(f'  [RAG] Found {len(context_parts)} relevant chunks')
    return {'context': context}

# Test
r = rag_node({'question': 'What is classification?', 'route': 'conceptual', 'context': '', 'tool_result': '', 'answer': ''})
print(f'Context length: {len(r["context"])} chars')

In [ ]:
import numpy as np

def tool_node(state):
    """Perform numerical calculations."""
    question = state['question']
    # Extract numbers from the question
    import re
    numbers = [float(x) for x in re.findall(r'-?\d+\.?\d*', question)]

    if not numbers:
        result_text = 'No numbers found in the question. Please provide numbers to calculate.'
    elif any(w in question.lower() for w in ['mean', 'average']):
        result_text = f'Mean = {np.mean(numbers):.4f}'
    elif any(w in question.lower() for w in ['std', 'standard deviation']):
        result_text = f'Standard deviation = {np.std(numbers, ddof=1):.4f}'
    elif any(w in question.lower() for w in ['median']):
        result_text = f'Median = {np.median(numbers):.4f}'
    elif any(w in question.lower() for w in ['sum', 'total']):
        result_text = f'Sum = {sum(numbers):.4f}'
    else:
        result_text = f'Numbers: {numbers}. Mean = {np.mean(numbers):.4f}, Std = {np.std(numbers, ddof=1):.4f}'

    print(f'  [Tool] Computed: {result_text}')
    return {'tool_result': result_text}

# Test
r = tool_node({'question': 'Calculate the mean of 10, 20, 30, 40', 'route': 'numerical', 'context': '', 'tool_result': '', 'answer': ''})
print(f'Result: {r["tool_result"]}')

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Data Science tutor. Answer concisely using the provided context if available.'),
    ('human', 'Context: {context}\nTool result: {tool_result}\nQuestion: {question}\n\nProvide a clear, educational answer.')
])

answer_chain = answer_prompt | llm | StrOutputParser()

def generate_node(state):
    """Generate an answer using LLM with available context."""
    answer = answer_chain.invoke({
        'context': state.get('context', 'No relevant documents found.')
        if state.get('route') == 'conceptual' else '',
        'tool_result': state.get('tool_result', '')
        if state.get('route') == 'numerical' else '',
        'question': state['question']
    })
    print(f'  [Generate] Answer length: {len(answer)} chars')
    return {'answer': answer}

# Test
r = generate_node({'question': 'What is classification?', 'route': 'conceptual',
    'context': 'Classification assigns categories to data.', 'tool_result': '', 'answer': ''})
print(f'Answer: {r["answer"][:150]}...')

---

## 7. Conditional Routing

The key feature of LangGraph is **conditional edges**. The `route_decision` function
examines the state and decides which node to visit next.

In [ ]:
def route_decision(state):
    """Decide which node to visit based on classification."""
    route = state.get('route', 'general')
    print(f'  [Route] Routing to: {route}')
    return route  # must match a key in the routing dict

print('Route decision function ready')

```mermaid
graph LR
    A[route_decision] -->|conceptual| B[rag_node]
    A -->|numerical| C[tool_node]
    A -->|general| D[generate_node]
```

---

## 8. Validation Loop

LangGraph supports **loops**. Here we add a validate node that checks the answer
and optionally routes back for improvement.

In [ ]:
def validate_node(state):
    """Validate the answer quality (simplified)."""
    answer = state.get('answer', '')
    attempts = state.get('attempts', 0)

    # Simple validation: is the answer long enough?
    if len(answer) < 20 and attempts < 2:
        print(f'  [Validate] Answer too short (attempt {attempts + 1}). Requesting improvement.')
        return {'validated': False, 'attempts': attempts + 1}
    else:
        print('  [Validate] Answer accepted.')
        return {'validated': True}

def after_validate(state):
    """Conditional edge after validation."""
    if state.get('validated', False):
        return 'end'
    else:
        return 'generate'  # loop back to generate

print('Validation logic ready')

---

## 9. Building the Complete Graph


In [ ]:
# Build the graph
builder = StateGraph(dict)

# Add nodes
builder.add_node('classify', classify_node)
builder.add_node('rag', rag_node)
builder.add_node('tool', tool_node)
builder.add_node('generate', generate_node)
builder.add_node('validate', validate_node)

# Add edges
builder.add_edge(START, 'classify')

# Conditional routing from classify
builder.add_conditional_edges(
    'classify',
    route_decision,
    {'conceptual': 'rag', 'numerical': 'tool', 'general': 'generate'}
)

# All paths lead to validate
builder.add_edge('rag', 'validate')
builder.add_edge('tool', 'validate')
builder.add_edge('generate', 'validate')

# Conditional edge from validate: either END or loop back to generate
builder.add_conditional_edges(
    'validate',
    after_validate,
    {'end': END, 'generate': 'generate'}
)

# Compile
ds_graph = builder.compile()
print('Graph compiled successfully!')

### Visualize the graph

LangGraph can generate a visual representation:


In [ ]:
# Print the graph structure
print(ds_graph.get_graph().draw_ascii())

---

## 10. Running the Graph


In [ ]:
# Test 1: Conceptual question (should route to RAG)
print('=== Test 1: Conceptual Question ===')
result = ds_graph.invoke({
    'question': 'What is classification in machine learning?',
    'route': '', 'context': '', 'tool_result': '',
    'answer': '', 'validated': False, 'attempts': 0
})
print(f'\nFinal answer: {result["answer"][:200]}')

In [ ]:
# Test 2: Numerical question (should route to Tool)
print('=== Test 2: Numerical Question ===')
result = ds_graph.invoke({
    'question': 'Calculate the mean of 15, 25, 35, 45, 55',
    'route': '', 'context': '', 'tool_result': '',
    'answer': '', 'validated': False, 'attempts': 0
})
print(f'\nFinal answer: {result["answer"][:200]}')

In [ ]:
# Test 3: General question (should go directly to LLM)
print('=== Test 3: General Question ===')
result = ds_graph.invoke({
    'question': 'What are the best practices for feature engineering?',
    'route': '', 'context': '', 'tool_result': '',
    'answer': '', 'validated': False, 'attempts': 0
})
print(f'\nFinal answer: {result["answer"][:200]}')

---

## 11. Local Ollama Implementation

LangGraph works with any LLM provider. Here is the same graph using Ollama:

In [ ]:
if ollama_available:
    ollama_llm = ChatOllama(model='llama3.2', temperature=0)
    ollama_answer_chain = answer_prompt | ollama_llm | StrOutputParser()

    def ollama_generate_node(state):
        answer = ollama_answer_chain.invoke({
            'context': state.get('context', ''),
            'tool_result': state.get('tool_result', ''),
            'question': state['question']
        })
        print(f'  [Ollama Generate] Answer length: {len(answer)} chars')
        return {'answer': answer}

    # Build Ollama graph
    ollama_builder = StateGraph(dict)
    ollama_builder.add_node('classify', classify_node)
    ollama_builder.add_node('rag', rag_node)
    ollama_builder.add_node('tool', tool_node)
    ollama_builder.add_node('generate', ollama_generate_node)
    ollama_builder.add_node('validate', validate_node)
    ollama_builder.add_edge(START, 'classify')
    ollama_builder.add_conditional_edges('classify', route_decision,
        {'conceptual': 'rag', 'numerical': 'tool', 'general': 'generate'})
    ollama_builder.add_edge('rag', 'validate')
    ollama_builder.add_edge('tool', 'validate')
    ollama_builder.add_edge('generate', 'validate')
    ollama_builder.add_conditional_edges('validate', after_validate,
        {'end': END, 'generate': 'generate'})

    ollama_graph = ollama_builder.compile()
    print('Ollama graph compiled!')

    result = ollama_graph.invoke({
        'question': 'What is clustering?',
        'route': '', 'context': '', 'tool_result': '',
        'answer': '', 'validated': False, 'attempts': 0
    })
    print(f'Ollama answer: {result["answer"][:200]}')
else:
    print('Ollama not available. Start with: ollama serve')
    print('You can still follow the code and logic above.')

---

## 12. Human-in-the-Loop

LangGraph supports **interrupts** where the graph pauses and waits for human input.
This is valuable for sensitive operations like:

- Executing database queries
- Deploying models
- Sending data externally

```mermaid
graph LR
    A[Agent decides] --> B[Interrupt]
    B --> C{Human approves?}
    C -->|Yes| D[Execute]
    C -->|No| E[Skip]
    D --> F[Continue]
    E --> F
```

```python
# Conceptual example (requires langgraph checkpointing for production use):
#
# from langgraph.checkpoint.memory import MemorySaver
# memory = MemorySaver()
# graph = builder.compile(checkpointer=memory, interrupt_before=['execute'])
#
# # Run until interrupt
# result = graph.invoke(initial_state, config={'thread_id': '1'})
# # result will contain 'interrupt' field
#
# # Human reviews and approves
# graph.update_state(config={'thread_id': '1'}, values={'approved': True})
# final = graph.invoke(None, config={'thread_id': '1'})
```

> **Note:** Full human-in-the-loop requires checkpointing (e.g., `MemorySaver`).
> The example above is conceptual. We will not implement it here to keep dependencies minimal.

---

## 13. Chain vs Agent vs Graph

| Aspect | Chain | Agent | Graph |
|--------|-------|-------|-------|
| **Flow** | Fixed linear | Dynamic (LLM decides) | Explicitly defined |
| **State** | Implicit | Implicit | Explicit (TypedDict) |
| **Loops** | No | Yes (unbounded) | Yes (bounded) |
| **Transparency** | High | Low (black box) | High |
| **Control** | Developer | LLM | Developer |
| **Best for** | Simple pipelines | Open-ended tasks | Complex workflows |

### When to use a graph?

- You need **conditional routing** based on data, not just the LLM
- You want **bounded loops** with clear exit conditions
- You need **human oversight** at specific steps
- You want **full visibility** into the workflow
- You need **persistence** across invocations

---

## 14. Security and Responsible AI

### Graph-specific concerns

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Infinite loops** | Graph keeps cycling without exit | Set max iterations, use `attempts` counter |
| **Unvalidated output** | Bad answers reach users | Always include validation nodes |
| **Tool abuse** | Agent calls dangerous tools | Restrict tool access per route |
| **Data leakage** | Sensitive data in state persists | Clear state between sessions |
| **Prompt injection** | Malicious input routes to dangerous path | Input validation at classify node |

### Best practices

1. **Always add validation nodes** before final output
2. **Limit loop iterations** to prevent runaway graphs
3. **Log all state transitions** for debugging
4. **Restrict tool access** based on routing decisions
5. **Use human-in-the-loop** for irreversible actions

---

## 15. Exercises

### Exercise 1: Add a Quiz Node

Add a new node `quiz_node` that generates quiz questions about a topic.
Route questions containing 'quiz' or 'test me' to this node.

### Exercise 2: Add Memory Counter

Modify the graph to track how many questions have been asked
and include a greeting node that changes its message based on the count.

### Exercise 3: Multi-hop Routing

Create a graph where some questions require TWO steps:
1. RAG retrieves information
2. A tool performs a calculation on the retrieved information

Hint: Use a conditional edge after RAG to decide if a tool is needed.

### Challenge: Research Assistant Graph

Build a complete research assistant with:
- Topic classification (theory / code / math / comparison)
- Different retrieval strategies per topic
- A validation loop that checks answer completeness
- Source citation in the final answer

---

## 16. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| **State** | Shared dictionary flowing through the graph |
| **Nodes** | Functions that read/write state |
| **Edges** | Connections between nodes |
| **Conditional edges** | Dynamic routing based on state |
| **Loops** | Controlled iteration with exit conditions |
| **Validation** | Quality checks before final output |
| **Human-in-the-loop** | Pausing for human approval |

### The complete 12-notebook stack

| # | Notebook | Core Skill |
|---|----------|------------|
| 01 | Introduction | LangChain basics |
| 02 | Models, Prompts, Messages | LLM interaction |
| 03 | LCEL and Chains | Pipeline composition |
| 04 | Embeddings and Vector Stores | Semantic search |
| 05 | RAG | Knowledge-grounded generation |
| 06 | Tools and Agents | Dynamic workflows |
| 07 | Capstone Project | Complete application |
| 08 | Advanced RAG | Production RAG techniques |
| 09 | Document Loading | Multi-format processing |
| 10 | SQL and Databases | Structured data interaction |
| 11 | Data Science Agents | Agent-based analysis |
| 12 | LangGraph | Stateful graph workflows |

### Next steps

- **LangSmith**: Observability and evaluation platform
- **LangGraph Platform**: Production deployment of graph workflows
- **Multi-agent systems**: Multiple agents collaborating
- **Persistent memory**: Long-term conversation memory
- **MCP (Model Context Protocol)**: Standardized tool integration